In [1]:
import numpy as pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Importar tabela de meta de alfabetização por município
df_meta_municipio = pd.read_parquet('../data/meta_alfabetizacao_municipio.parquet')
df_meta_municipio.head()

,ano,id_municipio,id_municipio_nome,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao
0,2023,4301750,Barão do Triunfo,Municipal,NaN,NaN,14.05,23.65,37.00,52.68,67.85,80.0,<NA>,NaN
1,2024,4301750,Barão do Triunfo,Municipal,4.40,NaN,14.05,23.65,37.00,52.68,67.85,80.0,0,92.59
2,2024,2406908,Lucrécia,Municipal,42.86,7.94,14.05,23.65,37.00,52.68,67.85,80.0,1,84.00
3,2023,2406908,Lucrécia,Municipal,4.40,7.94,14.05,23.65,37.00,52.68,67.85,80.0,0,82.14
4,2023,1718501,Recursolândia,Municipal,4.60,8.25,14.48,24.16,37.49,53.03,68.00,80.0,0,95.65


In [3]:
len(df_meta_municipio)

10704

In [4]:
# Verificar dados nulos
df_meta_municipio.isna().sum()

ano                          0
id_municipio                 0
id_municipio_nome            0
rede                         0
taxa_alfabetizacao         120
meta_alfabetizacao_2024    240
meta_alfabetizacao_2025      0
meta_alfabetizacao_2026      0
meta_alfabetizacao_2027      0
meta_alfabetizacao_2028      0
meta_alfabetizacao_2029      0
meta_alfabetizacao_2030      0
nivel_alfabetizacao        120
percentual_participacao    120
dtype: int64

In [5]:
# Entender relação entre dados nulos das colunas taxa_alfabetizacao e meta_alfabetizacao_2024
df_meta_municipio[df_meta_municipio['taxa_alfabetizacao'].isna()][['ano', 'id_municipio','taxa_alfabetizacao', 'meta_alfabetizacao_2024']]

,ano,id_municipio,taxa_alfabetizacao,meta_alfabetizacao_2024
0,2023,4301750,NaN,NaN
8,2023,2414456,NaN,NaN
13,2023,2406205,NaN,NaN
21,2023,2408706,NaN,NaN
26,2023,1200435,NaN,NaN
...,...,...,...,...
8844,2023,4216008,NaN,NaN
8846,2023,3147501,NaN,NaN
8848,2023,3151909,NaN,NaN
8850,2023,3166600,NaN,NaN


In [6]:
len(df_meta_municipio[df_meta_municipio['meta_alfabetizacao_2024'].isna()]['id_municipio'].unique())

120

Sempre que o município não tem meta em 2024, ele não tem taxa de alfabetização em 2023.

Para resolver esse problema, temos duas opções:
a) trabalhar apenas com os dados de taxa de alfabetização de 2024;
b) copiar a taxa de 2024 em 2023 e a meta de 2025 para 2024;

In [7]:
# Conferindo se a coluna taxa_alfabetizacao possui algum dado menor que 0 ou acima de 100, que podem indicar dados inconsistentes.
df_meta_municipio['taxa_alfabetizacao'].describe()

count    10584.000000
mean        61.774719
std         19.527540
min          4.400000
25%         47.600000
50%         62.445000
75%         76.742500
max        100.000000
Name: taxa_alfabetizacao, dtype: float64

In [8]:
# Comparar com a base município
df_municipio = pd.read_parquet('../data/municipio.parquet')
df_municipio.head()

,ano,id_municipio,id_municipio_nome,serie,chave_rede,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,1100031,Cabixi,2° ano do Ensino Fundamental,3,Municipal,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,1100072,Corumbiara,2° ano do Ensino Fundamental,3,Municipal,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,1100189,Pimenta Bueno,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,1101609,Theobroma,2° ano do Ensino Fundamental,3,Municipal,50.70,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,1101807,Vale do Paraíso,2° ano do Ensino Fundamental,3,Municipal,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Entendendo a quantidade de municípios em cada base:
print(f'A base Meta Municipio possui {len(df_meta_municipio['id_municipio'].unique())} municípios distintos.')
print(f'A base Municipio possui {len(df_municipio['id_municipio'].unique())} municípios distintos.')

A base Meta Municipio possui 5352 municípios distintos.
A base Municipio possui 5550 municípios distintos.


In [10]:
def compara_bases(df1, df2):
    bases_unidas = pd.merge(df1, df2, on=['ano', 'id_municipio', 'rede'], how='left')
    bases_unidas['taxa_diferente'] = (bases_unidas['taxa_alfabetizacao_x'] != bases_unidas['taxa_alfabetizacao_y']).astype(int)
    return bases_unidas
    

In [11]:

compara_taxa_alfabetizacao = compara_bases(df_meta_municipio, df_municipio[['ano', 'id_municipio', 'taxa_alfabetizacao', 'rede']])
print(f'Existem {len(compara_taxa_alfabetizacao[compara_taxa_alfabetizacao['taxa_diferente'] == 1])} linhas com taxas diferentes para um mesmo município em cada base.')

Existem 4716 linhas com taxas diferentes para um mesmo município em cada base.


In [12]:
# Listando os municípios com taxas diferentes para entender a diferença.
compara_taxa_alfabetizacao[compara_taxa_alfabetizacao['taxa_diferente'] == 1].sort_values('id_municipio')

,ano,id_municipio,id_municipio_nome,rede,taxa_alfabetizacao_x,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,taxa_alfabetizacao_y,taxa_diferente
6102,2023,1100015,Alta Floresta D'Oeste,Municipal,64.6,67.08,69.51,71.84,74.06,76.16,78.14,80.0,3,89.37,64.55,1
5740,2023,1100049,Cacoal,Municipal,62.5,65.39,68.16,70.81,73.33,75.70,77.92,80.0,3,84.44,62.51,1
4908,2023,1100056,Cerejeiras,Municipal,58.5,62.09,65.53,68.81,71.91,74.81,77.51,80.0,2,92.12,58.53,1
5811,2023,1100064,Colorado do Oeste,Municipal,62.9,65.67,68.39,70.98,73.45,75.78,77.96,80.0,3,86.98,62.85,1
9463,2023,1100080,Costa Marques,Municipal,80.7,80.00,80.00,80.00,80.00,80.00,80.00,80.0,5,91.06,80.68,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10472,2023,5221908,Varjão,Municipal,97.7,80.00,80.00,80.00,80.00,80.00,80.00,80.0,5,78.18,97.74,1
9488,2023,5222005,Vianópolis,Municipal,89.0,80.00,80.00,80.00,80.00,80.00,80.00,80.0,5,84.88,89.04,1
8524,2023,5222054,Vicentinópolis,Municipal,78.2,78.50,78.75,79.01,79.26,79.51,79.76,80.0,4,94.20,78.24,1
10438,2023,5222203,Vila Boa,Municipal,83.8,80.00,80.00,80.00,80.00,80.00,80.00,80.0,5,92.11,83.76,1


É possível notar que as notas são as taxas do dataframe Meta Município estão arredondando para 1 casa decimal, por isso a diferença.

In [13]:
compara_taxa_alfabetizacao[compara_taxa_alfabetizacao['taxa_diferente'] == 1].isna().sum()

ano                          0
id_municipio                 0
id_municipio_nome            0
rede                         0
taxa_alfabetizacao_x       120
meta_alfabetizacao_2024    120
meta_alfabetizacao_2025      0
meta_alfabetizacao_2026      0
meta_alfabetizacao_2027      0
meta_alfabetizacao_2028      0
meta_alfabetizacao_2029      0
meta_alfabetizacao_2030      0
nivel_alfabetizacao        120
percentual_participacao    120
taxa_alfabetizacao_y        50
taxa_diferente               0
dtype: int64

Apesar de constatarmos que as bases apresentam taxas de alfabetização semelhante, a base Município possui menos dados em branco.

In [14]:
# Entender a coluna rede
df_meta_municipio['rede'].value_counts()

rede
Municipal    10704
Name: count, dtype: int64

In [15]:
df_municipio['rede'].value_counts()

rede
Municipal                                         10896
Pública (Estadual e Municipal)                    10466
Estadual                                           2235
Total (Federal, Estadual, Municipal e Privada)      398
Name: count, dtype: int64

Como a base município possui mais redes listadas, preferimos usar a taxa de alfabetização desta base.

In [16]:
# Analisando as bases UF e Meta UF

df_uf = pd.read_parquet('../data/uf.parquet')
df_uf.head()

,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,PB,Paraíba,2° ano do Ensino Fundamental,Estadual,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,PR,Paraná,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,PE,Pernambuco,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
df_meta_uf = pd.read_parquet('../data/meta_alfabetizacao_uf.parquet')
df_meta_uf

,ano,sigla_uf,sigla_uf_nome,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2024,RR,Roraima,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,RR,Roraima,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,SE,Sergipe,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80.0,92.84
3,2023,SE,Sergipe,Pública,31.30,38.3,45.9,53.6,61.2,68.3,74.6,80.0,88.34
4,2025,SE,Sergipe,Pública,50.00,38.0,46.0,54.0,61.0,68.0,75.0,80.0,87.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,2023,PR,Paraná,Pública,73.12,74.2,75.2,76.2,77.2,78.2,79.1,80.0,86.18
77,2024,PR,Paraná,Pública,70.42,74.2,75.2,76.2,77.2,78.2,79.1,80.0,86.25
78,2025,CE,Ceará,Pública,84.00,80.0,80.0,80.0,80.0,80.0,80.0,80.0,96.00
79,2024,CE,Ceará,Pública,85.31,80.0,80.0,80.0,80.0,80.0,80.0,80.0,98.13


In [18]:
df_uf['rede'].value_counts()

rede
Municipal                                         49
Pública (Estadual e Municipal)                    49
Estadual                                          46
Total (Federal, Estadual, Municipal e Privada)     1
Name: count, dtype: int64

In [19]:
df_meta_uf['rede'].value_counts()

rede
Pública    81
Name: count, dtype: int64

In [20]:
df_meta_brasil = pd.read_parquet('../data/meta_alfabetizacao_brasil.parquet')
df_meta_brasil.head()

,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00


A princípio, não vamos usar as bases de meta para o modelo, porque só trazem informações da rede pública.

Vamos acrescentar então as informações que devem entrar no modelo.

In [21]:
# Os municípios não tem a UF vinculada nesta base. Então vamos juntar esta informação aqui.

df_diretorios_municipio = pd.read_parquet('../data/dir_municipio.parquet')
df_diretorios_municipio.head()

,id_municipio,sigla_uf
0,5101837,MT
1,1100205,RO
2,1100338,RO
3,1100809,RO
4,1101104,RO


In [22]:
acrescenta_dados_uf = pd.merge(
    df_municipio, df_diretorios_municipio, on=['id_municipio'], how='left'
)
print('Dados em branco na coluna sigla_uf: ', acrescenta_dados_uf['sigla_uf'].isna().sum())
print('Tamanho dataset antes do merge: ', len(df_municipio))
print('Tamanho dataset depois do merge: ', len(acrescenta_dados_uf))

Dados em branco na coluna sigla_uf:  0
Tamanho dataset antes do merge:  23995
Tamanho dataset depois do merge:  23995


In [23]:
# Acrescenta taxa de alfabetização da tabela uf

df_com_dados_de_taxa_de_alfabetizacao_da_uf = pd.merge(
    acrescenta_dados_uf, df_uf[['sigla_uf', 'ano', 'taxa_alfabetizacao', 'rede']], on=['sigla_uf', 'ano', 'rede'], how='left',
    suffixes=('_mun', '_uf')
)
print('Dados em branco na coluna taxa_alfabetizacao_uf: ', df_com_dados_de_taxa_de_alfabetizacao_da_uf['taxa_alfabetizacao_uf'].isna().sum())
print('Tamanho dataset antes do merge: ', len(acrescenta_dados_uf))
print('Tamanho dataset depois do merge: ', len(df_com_dados_de_taxa_de_alfabetizacao_da_uf))

Dados em branco na coluna taxa_alfabetizacao_uf:  2
Tamanho dataset antes do merge:  23995
Tamanho dataset depois do merge:  23995


In [24]:
# Vamos entender quais são os dados que ficaram em branco:

df_com_dados_de_taxa_de_alfabetizacao_da_uf[df_com_dados_de_taxa_de_alfabetizacao_da_uf['taxa_alfabetizacao_uf'].isna()]

,ano,id_municipio,id_municipio_nome,serie,chave_rede,rede,taxa_alfabetizacao_mun,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,sigla_uf,taxa_alfabetizacao_uf
14414,2024,5300108,Brasília,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),59.13,743.01,2.44,4.67,7.66,9.63,25.61,33.05,13.13,3.1,0.71,DF,NaN
14415,2024,5300108,Brasília,2° ano do Ensino Fundamental,2,Estadual,59.13,743.01,2.44,4.67,7.66,9.63,25.61,33.05,13.13,3.1,0.71,DF,NaN


Devido a natureza híbrida do Distrito Federal, ele não tem taxa de alfabetização como UF, apenas como município. Decidi copiar as notas da taxa de alfabetização municipal e usar na UF também.

In [25]:
# Copiando os dados do DF como município para 
condicao = (df_com_dados_de_taxa_de_alfabetizacao_da_uf['sigla_uf'] == 'DF') & (df_com_dados_de_taxa_de_alfabetizacao_da_uf['taxa_alfabetizacao_uf'].isna())
df_com_dados_de_taxa_de_alfabetizacao_da_uf.loc[condicao, 'taxa_alfabetizacao_uf'] = df_com_dados_de_taxa_de_alfabetizacao_da_uf.loc[condicao, 'taxa_alfabetizacao_mun']
df_com_dados_de_taxa_de_alfabetizacao_da_uf

,ano,id_municipio,id_municipio_nome,serie,chave_rede,rede,taxa_alfabetizacao_mun,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,sigla_uf,taxa_alfabetizacao_uf
0,2023,1100031,Cabixi,2° ano do Ensino Fundamental,3,Municipal,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RO,65.17
1,2023,1100072,Corumbiara,2° ano do Ensino Fundamental,3,Municipal,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RO,65.17
2,2023,1100189,Pimenta Bueno,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RO,64.60
3,2023,1101609,Theobroma,2° ano do Ensino Fundamental,3,Municipal,50.70,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RO,65.17
4,2023,1101807,Vale do Paraíso,2° ano do Ensino Fundamental,3,Municipal,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RO,65.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23990,2024,4319364,São Pedro das Missões,2° ano do Ensino Fundamental,3,Municipal,90.00,793.4553,0.0,0.0,0.0,10.0,0.00,20.00,0.0,60.00,10.00,RS,44.23
23991,2024,4319364,São Pedro das Missões,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),90.00,793.4553,0.0,0.0,0.0,10.0,0.00,20.00,0.0,60.00,10.00,RS,44.67
23992,2024,3146008,Ouro Fino,2° ano do Ensino Fundamental,2,Estadual,100.00,792.3100,0.0,0.0,0.0,0.0,10.00,20.00,10.0,60.00,0.00,MG,74.47
23993,2024,3529104,Marinópolis,2° ano do Ensino Fundamental,3,Municipal,100.00,804.6015,0.0,0.0,0.0,0.0,7.69,15.38,0.0,61.54,15.38,SP,55.71


In [26]:
# Baixar bases socioeconômicas para agregar a base do modelo
indice_analf = pd.read_parquet('../data/outras_fontes/indice_analfabetismo.parquet')
inse = pd.read_parquet('../data/outras_fontes/inse.parquet')
rendimento_per_capta = pd.read_parquet('../data/outras_fontes/rendimento_dom_per_capta.parquet')

In [27]:
# Entendendo as bases a serem usadas e fazendo a junção.
indice_analf.head()

,Sigla,Código,Município,indice_analf
0,AC,1200013,Acrelândia,11.65
1,AC,1200054,Assis Brasil,14.70
2,AC,1200104,Brasiléia,10.99
3,AC,1200138,Bujari,18.74
4,AC,1200179,Capixaba,15.73


In [28]:
# Precisamos renomear as colunas e padronizar o tipo.

indice_analf['id_municipio'] = indice_analf['Código']
indice_analf['id_municipio'] = indice_analf['id_municipio'].astype(str)
indice_analf.drop('Código', axis = 1, inplace=True)

In [30]:
agrega_bases_externas = pd.merge(df_com_dados_de_taxa_de_alfabetizacao_da_uf, indice_analf[['id_municipio', 'indice_analf']], how='left', on= 'id_municipio')

print('Dados em branco na coluna indice_analf: ', agrega_bases_externas['indice_analf'].isna().sum())
print('Tamanho dataset antes do merge: ', len(df_com_dados_de_taxa_de_alfabetizacao_da_uf))
print('Tamanho dataset depois do merge: ', len(agrega_bases_externas))

Dados em branco na coluna indice_analf:  0
Tamanho dataset antes do merge:  23995
Tamanho dataset depois do merge:  23995


In [31]:
# Adicionada a base de índice de analfabetismo, vamos entender a base com os dados socioeconômicos.

inse.head()

,NU_ANO_SAEB,CO_MUNICIPIO,TP_TIPO_REDE,MEDIA_INSE
0,2023,1100015,0,4.948
1,2023,1100015,2,4.981
2,2023,1100015,3,4.964
3,2023,1100015,5,4.948
4,2023,1100015,6,4.948


In [32]:
inse['NU_ANO_SAEB'].value_counts()

NU_ANO_SAEB
2023    28978
Name: count, dtype: int64

In [33]:
# Precisamos renomear as colunas e padronizar o tipo.

inse['id_municipio'] = inse['CO_MUNICIPIO'].astype(str)
inse['chave_rede'] = inse['TP_TIPO_REDE'].astype(str)
inse.drop('CO_MUNICIPIO', axis = 1, inplace=True)
inse.drop('TP_TIPO_REDE', axis = 1, inplace=True)
inse.head()


,NU_ANO_SAEB,MEDIA_INSE,id_municipio,chave_rede
0,2023,4.948,1100015,0
1,2023,4.981,1100015,2
2,2023,4.964,1100015,3
3,2023,4.948,1100015,5
4,2023,4.948,1100015,6


In [34]:
agrega_bases_externas_inse = pd.merge(agrega_bases_externas, inse, how='left', left_on=['ano','id_municipio', 'chave_rede'], right_on=['NU_ANO_SAEB','id_municipio', 'chave_rede'])

print('Dados em branco na coluna MEDIA_INSE: ', agrega_bases_externas_inse['MEDIA_INSE'].isna().sum())
print('Tamanho dataset antes do merge: ', len(agrega_bases_externas))
print('Tamanho dataset depois do merge: ', len(agrega_bases_externas_inse))

Dados em branco na coluna MEDIA_INSE:  12497
Tamanho dataset antes do merge:  23995
Tamanho dataset depois do merge:  23995


Como vimos acima, a base do INSE só contém dados do ano de 2023. Portanto, como nosso interesse é usar os dados de 2023 para prever o desempenho dos alunos de 2024, vamos eliminar os dados de 2024 da base.

In [35]:
# Vamos usar apenas o ano de 2023, então precisamos filtrar as linhas com o ano de 2023

agrega_bases_externas_inse_2023 = agrega_bases_externas_inse.query('ano == 2023')
print('Tamanho inicial do dataset de municípios: ', len(df_municipio))
print('Tamanho dataset depois do merge: ', len(agrega_bases_externas_inse_2023))

print('Dados em branco na coluna MEDIA_INSE: ', agrega_bases_externas_inse_2023['MEDIA_INSE'].isna().sum())

Tamanho inicial do dataset de municípios:  23995
Tamanho dataset depois do merge:  11547
Dados em branco na coluna MEDIA_INSE:  49


Consideramos que 49 linhas em branco em um dataset com 11547 é aceitável. Então, vamos só checar o restante das colunas antes de gravar esta base. 

In [36]:
agrega_bases_externas_inse_2023.isna().sum()

ano                            0
id_municipio                   0
id_municipio_nome              0
serie                          0
chave_rede                     0
rede                           0
taxa_alfabetizacao_mun         0
media_portugues                0
proporcao_aluno_nivel_0    11547
proporcao_aluno_nivel_1    11547
proporcao_aluno_nivel_2    11547
proporcao_aluno_nivel_3    11547
proporcao_aluno_nivel_4    11547
proporcao_aluno_nivel_5    11547
proporcao_aluno_nivel_6    11547
proporcao_aluno_nivel_7    11547
proporcao_aluno_nivel_8    11547
sigla_uf                       0
taxa_alfabetizacao_uf          0
indice_analf                   0
NU_ANO_SAEB                   49
MEDIA_INSE                    49
dtype: int64

In [37]:
# Vamos excluir as colunas de proporção, uma vez que o dado não foi gerado em 2023. Excluiremos, também, as colunas redundantes.

cols_to_drop = [
    'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 
    'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', 'NU_ANO_SAEB']
df_municipio_final = agrega_bases_externas_inse_2023.drop(columns=[c for c in cols_to_drop if c in agrega_bases_externas_inse_2023.columns])
df_municipio_final

,ano,id_municipio,id_municipio_nome,serie,chave_rede,rede,taxa_alfabetizacao_mun,media_portugues,sigla_uf,taxa_alfabetizacao_uf,indice_analf,MEDIA_INSE
0,2023,1100031,Cabixi,2° ano do Ensino Fundamental,3,Municipal,69.10,767.8763,RO,65.17,10.18,5.276
1,2023,1100072,Corumbiara,2° ano do Ensino Fundamental,3,Municipal,58.20,747.8918,RO,65.17,8.48,4.904
2,2023,1100189,Pimenta Bueno,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),69.73,762.4062,RO,64.60,6.76,5.015
3,2023,1101609,Theobroma,2° ano do Ensino Fundamental,3,Municipal,50.70,745.6802,RO,65.17,11.62,4.734
4,2023,1101807,Vale do Paraíso,2° ano do Ensino Fundamental,3,Municipal,55.69,752.3724,RO,65.17,10.50,4.807
...,...,...,...,...,...,...,...,...,...,...,...,...
11542,2023,5210406,Itaberaí,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),78.96,769.5056,GO,66.74,7.21,5.009
11543,2023,5215009,Nova Veneza,2° ano do Ensino Fundamental,3,Municipal,86.43,768.1884,GO,66.72,6.32,5.052
11544,2023,5216403,Paraúna,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),61.83,745.1936,GO,66.74,6.77,5.305
11545,2023,5218706,Rianápolis,2° ano do Ensino Fundamental,3,Municipal,76.56,761.5043,GO,66.72,9.59,5.177


In [38]:
df_municipio_final.to_parquet('../data/dados_modelo/municipio_agregado_outras_fontes.parquet')